In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# 1. LOAD DATA
df = pd.read_csv("NTC.csv", parse_dates=["published_date"]).rename(columns={"published_date": "Date"})

In [ ]:
df

In [ ]:
# 2. REMOVE DUPLICATE
dup_mask = df.duplicated(subset="Date", keep="first")
print(df[dup_mask])
print(f"Number of duplicates removed: {dup_mask.sum()}")
df = df[~dup_mask]

In [ ]:
# 3. SORT CHRONOLOGICALLY
df = df.sort_values("Date").reset_index(drop=True)

In [ ]:
df

In [ ]:
# 4. KEEP REQUIRED COLUMNS
df = df[[
        "Date",
        "open",
        "high",
        "low",
        "close",
        "per_change",
        "traded_quantity",
        "traded_amount",
        "status"
    ]
]
# 5. CONVERT NUMERIC COLUMNS
numeric_cols = [
    "open",
    "high",
    "low",
    "close",
    "per_change",
    "traded_quantity",
    "traded_amount"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df

In [ ]:
# 5. REMOVE INVLID ROWS
df = df.dropna(subset=["Date", "close"]).reset_index(drop=True)

In [ ]:
df

In [ ]:
# 6. DATA INFORMATION
print("Total observations:", len(df))
print("First date:", df["Date"].iloc[0])
print("Last date :", df["Date"].iloc[-1])
print("Last close:", df["close"].iloc[-1])

In [ ]:
# 7.PLOT ORIGINAL CLOSING PRICE
plt.figure(figsize=(15, 6))
plt.plot(df["Date"], df["close"], color="steelblue")
plt.title("NTC Closing Price over the time")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.grid(alpha=0.1)
plt.tight_layout()
plt.savefig("ntc_closing_price.png", dpi=120)
plt.show()
plt.close()

In [ ]:
# 8. AUGMENTED DICKEY-FULLER TEST
close = df["close"].astype(float)
adf_result = adfuller(close)

print("ADF TEST - ORIGINAL SERIES")
print("ADF Statistic:", adf_result[0])
print("p-value:", adf_result[1])#0.05
for key, value in adf_result[4].items():
    print(f"Critical Value ({key}): {value}")

In [ ]:
# 9. DETERMINE DIFFERENCING ORDER (d)

close_diff = df["close"].diff().dropna()
adf_diff = adfuller(close_diff)

if adf_diff[1] < 0.05:
    d = 1
    print(f"First difference is stationary (p={adf_diff[1]:.26f}). d = {d}")
else:
    close_diff2 = close_diff.diff().dropna()
    adf_diff2 = adfuller(close_diff2)

    if adf_diff2[1] < 0.05:
        d = 2
        print(f"Second difference is stationary (p={adf_diff2[1]:.26f}). d = {d}")
    else:
        close_diff3 = close_diff2.diff().dropna()
        adf_diff3 = adfuller(close_diff3)

        if adf_diff3[1] < 0.05:
            d = 3
            print(f"Third difference is stationary (p={adf_diff3[1]:.26f}). d = {d}")
        else:
            raise ValueError("Series is still non-stationary after third differencing.")

In [ ]:
# 10. TRAIN / VALIDATION / TEST SPLIT
train_frac, val_frac = 0.64, 0.80  # 64% train, 16% val, 20% test

train_end = int(len(df) * train_frac)
val_end = int(len(df) * val_frac)

train = df.iloc[:train_end].copy()
validation = df.iloc[train_end:val_end].copy()
test = df.iloc[val_end:].copy()

print("TRAIN / VALIDATION / TEST SPLIT")

print(f"Training observations  : {len(train)}")
print(f"Validation observations: {len(validation)}")
print(f"Testing observations   : {len(test)}")

print(f"\nTraining period  : {train['Date'].iloc[0]} to {train['Date'].iloc[-1]}")
print(f"Validation period: {validation['Date'].iloc[0]} to {validation['Date'].iloc[-1]}")
print(f"Testing period   : {test['Date'].iloc[0]} to {test['Date'].iloc[-1]}")

In [ ]:
# 11. WALK-FORWARD FORECAST FUNCTION (fast version, refit=False)

def walk_forward_forecast(train_series, test_series, order):
    history = list(train_series)
    model = ARIMA(history, order=order)
    model_fit = model.fit()

    predictions = []
    for actual in test_series:
        forecast = model_fit.forecast(steps=1)[0]
        predictions.append(forecast)
        history.append(actual)
        model_fit = model_fit.append([actual], refit=False)

    return np.array(predictions)

In [ ]:
# 12. ARIMA MODEL COMPARISON
#Select model using VALIDATION data only

candidate_models = [(p, d, q) for p in range(4) for q in range(4)]
model_results = []

print("ARIMA MODEL COMPARISON")

for order in candidate_models:
    p, d_value, q = order
    print(f"\nTesting ARIMA{order} ...")

    try:
        predictions = walk_forward_forecast(train["close"], validation["close"], order)
        actual_values = validation["close"].values

        mae = mean_absolute_error(actual_values, predictions)
        rmse = np.sqrt(mean_squared_error(actual_values, predictions))
        mape = np.mean(np.abs((actual_values - predictions) / actual_values)) * 100

        model_results.append({"p": p, "d": d_value, "q": q, "MAE": mae, "RMSE": rmse, "MAPE": mape})
        print(f"MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.4f}%")

    except Exception as e:
        print(f"ARIMA{order} failed: {e}")

In [ ]:
# 13. SELECT BEST MODEL (lowest RMSE on validation)

results_df = pd.DataFrame(model_results)
best_row = results_df.loc[results_df["RMSE"].idxmin()]
best_order = (int(best_row["p"]), int(best_row["d"]), int(best_row["q"]))

best_order

In [ ]:
# 14. FIT FINAL MODEL & FORECAST ON TEST SET
train_final = pd.concat([train, validation])
final_model = ARIMA(train_final["close"], order=best_order).fit()
test_forecast = final_model.forecast(steps=len(test))

result = pd.DataFrame({
    "Date": test["Date"].values,
    "Actual": test["close"].values,
    "Predicted": test_forecast.values
})

In [ ]:
# 15. WALK-FORWARD PREDICTIONS FOR SELECTED MODEL
predictions = walk_forward_forecast(train["close"], test["close"], best_order)

In [ ]:
# 16. CREATE RESULT TABLE
result = pd.DataFrame({
    "Date": test["Date"].values,
    "Actual": test["close"].values,
    "Predicted": predictions
})

result["Error"] = result["Actual"] - result["Predicted"]
result["Absolute_Error"] = result["Error"].abs()
result

In [ ]:
# 17. FINAL TEST METRICS
mae = mean_absolute_error(result["Actual"], result["Predicted"])
rmse = np.sqrt(mean_squared_error(result["Actual"], result["Predicted"]))
mape = np.mean(np.abs((result["Actual"] - result["Predicted"]) / result["Actual"])) * 100

print("FINAL TEST PERFORMANCE")
print(f"ARIMA model: {best_order}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAPE : {mape:.4f}%")

In [ ]:
# 18. LATEST 10 TEST PREDICTIONS
print("LATEST 10 TEST PREDICTIONS")
print("-" * 60)
print(result.tail(10).to_string(index=False))

In [ ]:
# 19. PLOT VALIDATION AND TEST PREDICTIONS

# Generate validation predictions using the selected ARIMA model
validation_predictions = walk_forward_forecast(
    train["close"],
    validation["close"],
    best_order
)

plt.figure(figsize=(15, 7))

# Training data
plt.plot(train["Date"],train["close"],color="steelblue",linewidth=1,label="Training")

# Validation actual
plt.plot(validation["Date"],validation["close"],color="black",linewidth=1.5,label="Validation Actual")

# Validation prediction
plt.plot(validation["Date"],validation_predictions,color="orange",linewidth=1,label="Validation Prediction")

# Test actual
plt.plot(test["Date"],result["Actual"],color="green",linewidth=1.5,label="Test Actual")

# Test prediction
plt.plot(test["Date"],result["Predicted"],color="red",linewidth=1,label="Test Prediction")

# Mark validation start
plt.axvline(validation["Date"].iloc[0],color="gray",linestyle="--",alpha=0.7,label="Validation Start")

# Mark test start
plt.axvline(test["Date"].iloc[0],color="purple",linestyle="--",alpha=0.7,label="Test Start")

plt.title(f"NTC Closing Price - ARIMA{best_order}")
plt.xlabel("Date")
plt.ylabel("Closing Price")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

# Save before showing
plt.savefig("ntc_validation_test_predictions.png", dpi=120)

plt.show()
plt.close()


In [ ]:
# 20. PLOT TEST DATA ONLY
plt.figure(figsize=(18, 8))

plt.plot(test["Date"],result["Actual"],color="black",linewidth=1,label="Actual")

plt.plot(test["Date"],result["Predicted"],color="red",linewidth=1,label="ARIMA Prediction")

plt.title(f"NTC Closing Price - Test Data - ARIMA{best_order}", fontsize=16)

plt.xlabel("Date", fontsize=12)
plt.ylabel("Closing Price", fontsize=12)

plt.legend(fontsize=11)
plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig("ntc_test_predictions.png", dpi=150)
plt.show()
plt.close()

In [ ]:
# 21. FIT MODEL ON ALL AVAILABLE DATA
all_history = list(df["close"].astype(float).values)
final_model = ARIMA(all_history, order=best_order)
final_fitted = final_model.fit()

In [ ]:
# 22. PREDICT NEXT OBSERVATION
future_prediction = final_fitted.forecast(steps=1)[0]

In [ ]:
# 23. FIND NEXT WEEKDAY
last_date = df["Date"].iloc[-1]
next_date = last_date + pd.Timedelta(days=1)

while next_date.weekday() >= 5:
    next_date += pd.Timedelta(days=1)

In [ ]:
# 24. DISPLAY FUTURE PREDICTION
print("Last actual date      :", last_date)
print("Last actual close     :", df["close"].iloc[-1])
print("Next predicted date   :", next_date)
print("Predicted closing price:", round(future_prediction, 4))
# 25. SAVE TEST PREDICTIONS
result.to_csv("NTC_ARIMA_test_predictions.csv", index=False)

In [ ]:
# 26. SAVE FUTURE FORECAST
future_result = pd.DataFrame({
    "Last_Actual_Date": [last_date],
    "Last_Actual_Close": [df["close"].iloc[-1]],
    "Predicted_Date": [next_date],
    "Predicted_Close": [future_prediction],
    "Model": [f"ARIMA{best_order}"]
})
future_result.to_csv("NTC_ARIMA_future_forecast.csv", index=False)

print("\nFiles saved:")
print("1. NTC_ARIMA_test_predictions.csv")
print("2. NTC_ARIMA_future_forecast.csv")
print("3. ntc_closing_price.png")
print("4. ntc_test_predictions.png")